# verify_artifacts

Post-hoc verification of the pipeline from the artifacts on Drive. No GPU, no regeneration,
no scoring. Every check reads files that already exist and re-derives a figure asserted
elsewhere in the project.

Covers the properties the generation and training notebooks assert about themselves but do
not record: adapter attachment, the musical-keyboard cue rule, `save_state` behaviour, soft-prompt drift at the final step, filler length-match headroom, the rotary bound the soft-prompt runs sit inside, key alignment across every score file, pre-registration integrity, and the val-loss series transcribed by hand into the three check notebooks.

Run top to bottom. Each section prints its own PASS/FAIL line; section 13 aggregates.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, re, math, glob, struct, random, hashlib
from collections import defaultdict
import numpy as np
from scipy import stats as st

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SC  = f"{DRIVE}/scores"
EXP = f"{DRIVE}/lora/experiments"
SP  = f"{DRIVE}/soft_prompt/experiments"

RESULTS = []

def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail))
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" - {detail}" if detail else ""))

def rows(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def load_scores(name):
    return list(rows(f"{SC}/{name}"))

def ov(r):
    return math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))

def keyof(r):
    return r.get("uid", r.get("key"))

def paired(a, b):
    A = {keyof(r): r for r in load_scores(a)}
    B = {keyof(r): r for r in load_scores(b)}
    ks = [k for k in A if k in B]
    d = np.array([ov(A[k]) - ov(B[k]) for k in ks])
    return d.mean(), d.std(ddof=1), len(d)

print(DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data


## 1. Domain subset and the musical-keyboard cue

`data.ipynb` records a residual false-positive rate without the cue rule that produced it.
The rule is the plain substring `music`, which also matches *musical* and *musician*.
Re-derived here against the frozen splits.

In [2]:
KEYWORDS = ["desktop computer", "computer setup", "laptop", "tablet", "smartphone",
            "mobile phone", "cell phone", "headphones", "earphones", "earbuds",
            "keyboard", "computer mouse", "game controller", "gaming controller",
            "gaming console", "vr headset", "virtual reality headset"]
OTHER = [k for k in KEYWORDS if k != "keyboard"]
CUE = "music"

splits = {s: list(rows(f"{DRIVE}/{s}.jsonl")) for s in ("train", "val", "test")}
allrows = splits["train"] + splits["val"] + splits["test"]

def kb_only(instr):
    t = (instr or "").lower()
    return "keyboard" in t and not any(k in t for k in OTHER)

def musical(instr):
    return kb_only(instr) and CUE in (instr or "").lower()

n_kb = sum(1 for r in allrows if kb_only(r["instruction"]))
cue_rows = [r for r in allrows if musical(r["instruction"])]
t_rows = [r for r in splits["test"] if musical(r["instruction"])]
t_tgts = sorted({r["output_image"] for r in t_rows})

print(f"subset rows                             {len(allrows)}")
print(f"'keyboard', no other device keyword     {n_kb}")
print(f"... of which carry the cue '{CUE}'       {len(cue_rows)}  ({100*len(cue_rows)/len(allrows):.1f}% of subset)")
print(f"test split                              {len(t_rows)} rows from {len(t_tgts)} target(s): {t_tgts}")

check("subset is 40,479 rows", len(allrows) == 40479, str(len(allrows)))
check("musical-keyboard count is 408", len(cue_rows) == 408, str(len(cue_rows)))
check("test split is 7 rows from 1 target", len(t_rows) == 7 and len(t_tgts) == 1,
      f"{len(t_rows)} rows / {len(t_tgts)} target")

subset rows                             40479
'keyboard', no other device keyword     892
... of which carry the cue 'music'       408  (1.0% of subset)
test split                              7 rows from 1 target(s): ['triple/target3/11_3686.png']
  [PASS] subset is 40,479 rows - 40479
  [PASS] musical-keyboard count is 408 - 408
  [PASS] test split is 7 rows from 1 target - 7 rows / 1 target


### 1b. Dropping those rows moves no verdict

Every comparison is uid-paired, so the flagged rows appear on both sides of every contrast; excluding them changes no verdict, as the deltas below show.

In [3]:
bad = {r["output_image"] + "|" + "|".join(sorted(r["input_images"])) for r in t_rows}

def delta_excl(arm, base, exclude):
    A = {r["uid"]: r for r in load_scores(arm)}
    B = {r["uid"]: r for r in load_scores(base)}
    ks = [k for k in A if k in B and (k not in bad if exclude else True)]
    return float(np.mean([ov(A[k]) - ov(B[k]) for k in ks])), len(ks)

EXPECT = {"lora_r8@1000": ("test__lora_r8_step1000.jsonl", -0.120, -0.118),
          "QLoRA@2000":   ("test__qlora_r8_step2000.jsonl", -0.080, -0.075),
          "sp_iii@2000":  ("test__sp_iii_p139_step2000.jsonl", -0.826, -0.827)}

ok = True
for lbl, (fn, e_in, e_out) in EXPECT.items():
    m0, n0 = delta_excl(fn, "test__baseline.jsonl", False)
    m1, n1 = delta_excl(fn, "test__baseline.jsonl", True)
    print(f"  {lbl:<14} with {m0:+.3f} (n={n0})  ->  without {m1:+.3f} (n={n1})")
    ok = ok and abs(m0 - e_in) < 5e-4 and abs(m1 - e_out) < 5e-4

check("robustness deltas reproduce", ok)

  lora_r8@1000   with -0.120 (n=488)  ->  without -0.118 (n=481)
  QLoRA@2000     with -0.080 (n=488)  ->  without -0.075 (n=481)
  sp_iii@2000    with -0.826 (n=488)  ->  without -0.827 (n=481)
  [PASS] robustness deltas reproduce


## 2. Split reproducibility

`data.ipynb` builds `groups` as an insertion-ordered dict, so the target sets are
deterministic under seed 42. Row order is not: `dump()` iterates a set.

Skipped if `grit-entity-new.jsonl` is not on Drive.

In [4]:
GRIT = f"{DRIVE}/grit-entity-new.jsonl"
frozen = {s: {r["output_image"] for r in splits[s]} for s in splits}

if not os.path.exists(GRIT):
    print("grit-entity-new.jsonl not on Drive - section skipped")
else:
    pat = re.compile("|".join(re.escape(k) for k in KEYWORDS), re.IGNORECASE)
    KEEP = {"single", "double", "triple"}
    random.seed(42)
    groups = defaultdict(list)
    for row in rows(GRIT):
        if not pat.search(str(row.get("instruction", ""))):
            continue
        imgs, out = row.get("input_images", []), row.get("output_image", "")
        dirs = {p.split("/")[0] for p in imgs + [out] if p}
        if not dirs or not dirs.issubset(KEEP):
            continue
        groups[out].append(row)
    targets = list(groups.keys())
    random.shuffle(targets)
    rec = {"test": set(targets[:100]), "val": set(targets[100:120]), "train": set(targets[120:])}
    for s in ("train", "val", "test"):
        print(f"  {s:<6} reproduced {len(rec[s])} targets, identical to frozen: {rec[s] == frozen[s]}")
    check("target sets reproduce from seed 42", all(rec[s] == frozen[s] for s in rec))

check("no target overlap between splits",
      not (frozen["train"] & frozen["val"] or frozen["train"] & frozen["test"]
           or frozen["val"] & frozen["test"]))

spread = {s: {k: sum(1 for r in splits[s] if len(r["input_images"]) == k) for k in (1, 2, 3)}
          for s in ("train", "val", "test")}
for s in ("train", "val", "test"):
    n = spread[s]
    print(f"  {s:<6} reference spread  1-ref {n[1]} | 2-ref {n[2]} | 3-ref {n[3]}")
total = {k: sum(spread[s][k] for s in spread) for k in (1, 2, 3)}
check("reference spread per split sums to the subset spread",
      total == {1: 20122, 2: 16150, 3: 4207}, f"{total[1]}/{total[2]}/{total[3]}")

  train  reproduced 8059 targets, identical to frozen: True
  val    reproduced 20 targets, identical to frozen: True
  test   reproduced 100 targets, identical to frozen: True
  [PASS] target sets reproduce from seed 42
  [PASS] no target overlap between splits
  train  reference spread  1-ref 19830 | 2-ref 15917 | 3-ref 4146
  val    reference spread  1-ref 49 | 2-ref 39 | 3-ref 10
  test   reference spread  1-ref 243 | 2-ref 194 | 3-ref 51
  [PASS] reference spread per split sums to the subset spread - 20122/16150/4207


## 3. Few-shot exemplars

`fewshot_exemplars.jsonl` is write-once. Re-derived byte-for-byte from the frozen train
split and the grit source, and the mismatched pool is checked disjoint from every split.

In [5]:
def strip_refs(instruction):
    return " ".join(p for p in re.split(r"(?<=\.)\s+", instruction) if "<img>" not in p).strip()

FX = f"{DRIVE}/fewshot_exemplars.jsonl"
fx = list(rows(FX))
pat_kw = re.compile("|".join(re.escape(k) for k in KEYWORDS), re.I)

dom_all = [e for r in fx for e in r["domain"]]
mis_all = [e for r in fx for e in r["mismatched"]]
n_dev = sum(1 for e in mis_all if pat_kw.search(e))

print(f"exemplar rows {len(fx)}")
print(f"  domain     {len(dom_all)} instances, {len(set(dom_all))} unique")
print(f"  mismatched {len(mis_all)} instances, {len(set(mis_all))} unique, device keywords: {n_dev}")
check("no device keyword in the mismatched pool", n_dev == 0, str(n_dev))

if os.path.exists(GRIT):
    dom_by_t = defaultdict(list)
    for r in splits["train"]:
        dom_by_t[r["output_image"]].append(strip_refs(r["instruction"]))
    dom_targets = list(dom_by_t.keys())

    mis_by_t = defaultdict(list)
    for r in rows(GRIT):
        if r["output_image"].split("/")[0] in ("character", "human"):
            continue
        if not pat_kw.search(r["instruction"]):
            mis_by_t[r["output_image"]].append(strip_refs(r["instruction"]))
        if sum(len(v) for v in mis_by_t.values()) >= 20000:
            break
    mis_targets = list(mis_by_t.keys())

    overlap = {s: len(set(mis_targets) & frozen[s]) for s in frozen}
    print(f"  mismatched pool: {len(mis_targets)} targets, overlap {overlap}")
    check("mismatched pool disjoint from train/val/test", sum(overlap.values()) == 0, str(overlap))

    random.seed(42)
    lines = []
    for r in splits["test"]:
        uid = r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
        dom = [random.choice(dom_by_t[t]) for t in random.sample(dom_targets, 5)]
        mis = [random.choice(mis_by_t[t]) for t in random.sample(mis_targets, 5)]
        lines.append(json.dumps({"uid": uid, "domain": dom, "mismatched": mis}))
    rebuilt = "\n".join(lines) + "\n"
    orig = open(FX, encoding="utf-8").read()
    check("fewshot_exemplars.jsonl reproduces byte-for-byte", rebuilt == orig,
          hashlib.md5(orig.encode()).hexdigest())

exemplar rows 488
  domain     2440 instances, 2111 unique
  mismatched 2440 instances, 1997 unique, device keywords: 0
  [PASS] no device keyword in the mismatched pool - 0
  mismatched pool: 5583 targets, overlap {'train': 0, 'val': 0, 'test': 0}
  [PASS] mismatched pool disjoint from train/val/test - {'train': 0, 'val': 0, 'test': 0}
  [PASS] fewshot_exemplars.jsonl reproduces byte-for-byte - 306ea78f7046f092f89b3a14b6c0ec82


## 4. Filler control headroom

The length-matched filler truncates a fixed neutral string to the token count of the domain
block. If the block ever exceeded the neutral pool the slice would return a short prompt
with no error, silently breaking the length match. Asserted in tokens, the unit the filler is truncated by. The character margin is reported
alongside as a descriptive secondary figure; it is not the property the control relies on,
and it is wide (1,309) where the token margin is zero.

In [6]:
HEAD = "Here are descriptions of example images:\n"
NEUTRAL = "The following text is neutral placeholder content and describes nothing in particular. " * 60

def block(exs, k):
    return "".join(f"{i+1}. {e}\n" for i, e in enumerate(exs[:k]))

for K in (2, 5):
    L = [len(HEAD + block(r["domain"], K)) for r in fx]
    print(f"  K={K} domain prepend: mean {sum(L)/len(L):.0f}, max {max(L)} chars")

L5 = [len(HEAD + block(r["domain"], 5)) for r in fx]
print(f"  NEUTRAL pool {len(NEUTRAL)} chars, char headroom at the longest row {len(NEUTRAL) - max(L5)}")

# the filler is truncated by token count, so tokens are the property the control depends on
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained("OmniGen2/OmniGen2", subfolder="processor")
T5 = [len(_tok(HEAD + block(r["domain"], 5)).input_ids) for r in fx]
pool_tok = len(_tok(NEUTRAL).input_ids)
over_tok = sum(1 for x in T5 if x > pool_tok)
print(f"  NEUTRAL pool {pool_tok} tokens, token headroom at the longest row {pool_tok - max(T5)}")
check("no row exceeds the neutral pool in tokens", over_tok == 0, f"{over_tok}/{len(fx)}")

  K=2 domain prepend: mean 1278, max 1812 chars
  K=5 domain prepend: mean 3151, max 3911 chars
  NEUTRAL pool 5220 chars, char headroom at the longest row 1309
  NEUTRAL pool 781 tokens, token headroom at the longest row 0
  [PASS] no row exceeds the neutral pool in tokens - 0/488


## 5. `save_state` never fired in the soft-prompt runs

Patch S inserts the prompt write above the shipped checkpoint block and leaves its
condition alone, so `accelerator.save_state` stayed gated on `checkpointing_steps`, which
is set beyond the run length. Read off the console logs.

In [7]:
for run in ("sp_i_p668", "sp_ii_p668", "sp_iii_p139"):
    log = f"{SP}/{run}/console.log"
    if not os.path.exists(log):
        print(f"  {run}: console.log missing")
        continue
    txt = open(log, errors="replace").read()
    saved = len(re.findall(r"\[sp\] saved prompt-\d+\.pt", txt))
    state = len(re.findall(r"Saving current state|save_state", txt))
    fresh = "does not exist. Starting a new training run" in txt
    ckdirs = len(glob.glob(f"{SP}/{run}/checkpoint-*"))
    steps = sorted(int(x) for x in re.findall(r"\[sp\] step (\d+) grad_norm", txt))
    gaps = sorted(set(steps[i+1] - steps[i] for i in range(len(steps) - 1)))
    print(f"  {run}: prompt writes {saved} | state-save log lines {state} | "
          f"state dirs on disk {ckdirs} | fresh start {fresh} | monitor gaps {gaps}")
    check(f"{run}: six prompt tensors, no accelerate state",
          saved == 6 and state == 0 and ckdirs == 0)
    check(f"{run}: single session, no resume", fresh and gaps == [50])

  sp_i_p668: prompt writes 6 | state-save log lines 0 | state dirs on disk 0 | fresh start True | monitor gaps [50]
  [PASS] sp_i_p668: six prompt tensors, no accelerate state
  [PASS] sp_i_p668: single session, no resume
  sp_ii_p668: prompt writes 6 | state-save log lines 0 | state dirs on disk 0 | fresh start True | monitor gaps [50]
  [PASS] sp_ii_p668: six prompt tensors, no accelerate state
  [PASS] sp_ii_p668: single session, no resume
  sp_iii_p139: prompt writes 6 | state-save log lines 0 | state dirs on disk 0 | fresh start True | monitor gaps [50]
  [PASS] sp_iii_p139: six prompt tensors, no accelerate state
  [PASS] sp_iii_p139: single session, no resume


## 6. Adapter attachment

An adapter that failed to attach would reproduce the base config, so its per-row deltas
against the matching no-adapter run would sit at the identical-config floor
(sigma_d 0.966, 95% CI [0.895, 1.050]) with a mean near zero.

Section 6b supports attachment: divergence scales with adapter magnitude, which a non-attached run could not do.

In [8]:
FLOOR_HI = 1.050
LEGS = [("test", "test__baseline.jsonl",
         ["test__lora_r4_step1000.jsonl", "test__lora_r8_step1000.jsonl",
          "test__lora_r16_step1000.jsonl", "test__lora_r4.jsonl", "test__lora_r8.jsonl",
          "test__lora_r16.jsonl", "test__qlora_r8_step2000.jsonl"]),
        ("zero-ref", "test__zero_ref.jsonl", ["test__lora_r8_step1000_zero_ref.jsonl"]),
        ("omnicontext", "omnicontext__baseline.jsonl",
         ["omnicontext__lora_r8_step1000.jsonl", "omnicontext__qlora_r8_step2000.jsonl",
          "omnicontext__sp_iii_p139_step2000.jsonl"])]

low = []
for leg, base, arms in LEGS:
    print(f"  -- {leg} --")
    for a in arms:
        if not os.path.exists(f"{SC}/{a}"):
            continue
        m, s, n = paired(a, base)
        if s <= FLOOR_HI:
            low.append(a)
        flag = "" if s > FLOOR_HI else "   <- at floor, see 6b"
        print(f"     {a.replace('.jsonl', ''):<42} mean {m:+.3f}  sigma_d {s:.3f}  n {n}{flag}")

check("every test/omnicontext adapter run diverges from its base run", not low, str(low))

  -- test --
     test__lora_r4_step1000                     mean -0.115  sigma_d 1.180  n 488
     test__lora_r8_step1000                     mean -0.120  sigma_d 1.191  n 488
     test__lora_r16_step1000                    mean -0.111  sigma_d 1.216  n 488
     test__lora_r4                              mean -0.447  sigma_d 1.806  n 488
     test__lora_r8                              mean -0.368  sigma_d 1.739  n 488
     test__lora_r16                             mean -0.437  sigma_d 1.731  n 488
     test__qlora_r8_step2000                    mean -0.080  sigma_d 1.376  n 488
  -- zero-ref --
     test__lora_r8_step1000_zero_ref            mean +0.064  sigma_d 1.279  n 488
  -- omnicontext --
     omnicontext__lora_r8_step1000              mean -0.071  sigma_d 1.154  n 400
     omnicontext__qlora_r8_step2000             mean +0.089  sigma_d 1.229  n 400
     omnicontext__sp_iii_p139_step2000          mean -0.815  sigma_d 2.108  n 400
  [PASS] every test/omnicontext adapter run dive

### 6b. Divergence scales with adapter magnitude

Reads the converted adapters directly. `rms(A)*rms(B)` is a proxy for the size of the
update the adapter applies. A run whose adapter never attached would sit at the floor
regardless of its checkpoint's magnitude.

In [9]:
def st_tensors(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        hdr = json.loads(f.read(n))
        base = 8 + n
        f.seek(0)
        blob = f.read()
    out = {}
    for k, v in hdr.items():
        if k == "__metadata__":
            continue
        s, e = v["data_offsets"]
        raw = blob[base + s:base + e]
        if v["dtype"] == "F32":
            out[k] = np.frombuffer(raw, dtype=np.float32)
        elif v["dtype"] == "BF16":
            out[k] = (np.frombuffer(raw, dtype=np.uint16).astype(np.uint32) << 16).view(np.float32)
    return out

def adapter_stats(run, ck):
    p = f"{EXP}/{run}/checkpoint-{ck}/transformer_lora/pytorch_lora_weights.safetensors"
    if not os.path.exists(p):
        return None
    t = st_tensors(p)
    A = np.concatenate([v for k, v in t.items() if "lora_A" in k])
    B = np.concatenate([v for k, v in t.items() if "lora_B" in k])
    rms_a = float(np.sqrt((A ** 2).mean()))
    rms_b = float(np.sqrt((B ** 2).mean()))
    return rms_a * rms_b, len(t), rms_b

GRID = ([("lora_r8_early", s, f"val__lora_r8_early_step{s}.jsonl") for s in (250, 500, 750, 1000)]
        + [("lora_r4", s, f"val__lora_r4_step{s}.jsonl") for s in (1000, 2000, 3000, 4000)]
        + [("lora_r8", s, f"val__lora_r8_step{s}.jsonl") for s in (1000, 2000, 3000, 4000)]
        + [("lora_r16", s, f"val__lora_r16_step{s}.jsonl") for s in (1000, 2000, 3000, 4000)]
        + [("qlora_r8", s, f"val__qlora_r8_step{s}.jsonl") for s in (500, 1000, 1500, 2000)])

X, Y, zero_B = [], [], []
print(f"  {'run':<16}{'ck':>6}{'tensors':>9}{'rms(A)*rms(B)':>16}{'sigma_d vs val base':>22}")
for run, ck, fn in GRID:
    stats_ = adapter_stats(run, ck)
    if stats_ is None or not os.path.exists(f"{SC}/{fn}"):
        continue
    scale, ntens, rms_b = stats_
    if rms_b == 0:
        zero_B.append((run, ck))
    _, sd, _ = paired(fn, "val__baseline_val.jsonl")
    X.append(scale)
    Y.append(sd)
    print(f"  {run:<16}{ck:>6}{ntens:>9}{scale:>16.3e}{sd:>22.3f}")

if len(X) > 3:
    pr = st.pearsonr(np.log(X), Y)
    sp = st.spearmanr(X, Y)
    print(f"\n  Pearson(log scale, sigma_d) {pr[0]:+.3f} (p={pr[1]:.2e}) | "
          f"Spearman {sp.statistic:+.3f} (p={sp.pvalue:.2e})")
    check("divergence scales with adapter magnitude", pr[0] > 0.4 and pr[1] < 0.05)

check("no adapter has an all-zero lora_B", not zero_B, str(zero_B))

  run                 ck  tensors   rms(A)*rms(B)   sigma_d vs val base
  lora_r8_early      250      304       2.124e-06                 1.046
  lora_r8_early      500      304       8.252e-06                 0.973
  lora_r8_early      750      304       1.624e-05                 1.074
  lora_r8_early     1000      304       2.355e-05                 1.237
  lora_r4           1000      304       4.694e-05                 1.070
  lora_r4           2000      304       9.601e-05                 1.411
  lora_r4           3000      304       1.330e-04                 1.877
  lora_r4           4000      304       1.621e-04                 1.809
  lora_r8           1000      304       2.350e-05                 1.128
  lora_r8           2000      304       4.813e-05                 1.604
  lora_r8           3000      304       6.624e-05                 1.795
  lora_r8           4000      304       8.036e-05                 1.790
  lora_r16          1000      304       1.178e-05               

## 7. Reproducibility seam and adapter endpoints

`lora_r8_early` re-runs the first 1000 steps with seed 2233 and every other setting fixed,
so step 1000 overlaps the main run. The seam is argued elsewhere from scores; here it is
settled at the weight level.

The QLoRA comparison is the same measurement applied across precisions.

In [10]:
def lora_B(run, ck):
    p = f"{EXP}/{run}/checkpoint-{ck}/transformer_lora/pytorch_lora_weights.safetensors"
    t = st_tensors(p)
    return np.concatenate([v for k, v in sorted(t.items()) if "lora_B" in k])

try:
    a, b = lora_B("lora_r8", 1000), lora_B("lora_r8_early", 1000)
    corr = float(np.corrcoef(a, b)[0, 1])
    mad = float(np.abs(a - b).mean())
    rms = float(np.sqrt((a ** 2).mean()))
    print(f"  seam  lora_r8@1000 vs lora_r8_early@1000: corr {corr:.4f}, "
          f"mean|delta| {mad:.3e} against rms {rms:.3e} ({100*mad/rms:.2f}% relative)")
    check("reproducibility seam: the two runs land on nearly identical adapter endpoints", corr > 0.999)
except (FileNotFoundError, OSError) as e:
    print("  seam skipped:", e)

try:
    q, l = lora_B("qlora_r8", 2000), lora_B("lora_r8", 2000)
    print(f"  qlora_r8@2000 vs lora_r8@2000 (same rank, same step): "
          f"corr {float(np.corrcoef(q, l)[0, 1]):.4f}")
    print("  materially different adapter endpoints at matched rank and step; the task metric is")
    print("  equivalent anyway. The arms differ in precision and in sharding (LoRA under FSDP,")
    print("  QLoRA without), so this does not isolate NF4, and endpoints do not observe the path.")
except (FileNotFoundError, OSError) as e:
    print("  qlora comparison skipped:", e)

  seam  lora_r8@1000 vs lora_r8_early@1000: corr 1.0000, mean|delta| 9.851e-07 against rms 1.881e-04 (0.52% relative)
  [PASS] reproducibility seam: the two runs land on nearly identical adapter endpoints
  qlora_r8@2000 vs lora_r8@2000 (same rank, same step): corr 0.7722
  materially different adapter endpoints at matched rank and step; the task metric is
  equivalent anyway. The arms differ in precision and in sharding (LoRA under FSDP,
  QLoRA without), so this does not isolate NF4, and endpoints do not observe the path.


## 8. Soft-prompt drift at the final step

The console monitor logs every 50 steps and therefore stops at 2950. Final drift is read
from the checkpoint tensors.

In [11]:
try:
    import torch
    EXPECT_DRIFT = {"sp_i_p668": 187.3, "sp_ii_p668": 190.2, "sp_iii_p139": 462.0}
    for run in ("sp_i_p668", "sp_ii_p668", "sp_iii_p139"):
        f1 = f"{SP}/{run}/prompt-3000.pt"
        log = f"{SP}/{run}/console.log"
        if not (os.path.exists(f1) and os.path.exists(log)):
            continue
        d1 = torch.load(f1, map_location="cpu")
        # variants (i) and (ii) store the tensor under 'soft_prompt'; variant (iii) under 'prefix'
        t1 = d1["soft_prompt" if "soft_prompt" in d1 else "prefix"].float()
        m = re.search(r"\[sp\] step 0 grad_norm \S+ (?:prompt|prefix)_norm ([0-9.]+)",
                      open(log, errors="replace").read())
        init = float(m.group(1))
        n1 = float(t1.norm())
        drift = 100 * (n1 - init) / init
        print(f"  {run:<14} P={d1.get('P')}  init {init:.4f} -> step3000 {n1:.4f}"
              f"   drift {drift:+.1f}%   params {t1.numel():,}")
        check(f"{run}: final drift is {EXPECT_DRIFT[run]:+.1f}%",
              abs(drift - EXPECT_DRIFT[run]) < 0.1, f"{drift:+.1f}%")
except ImportError:
    print("  torch unavailable - section skipped")

  sp_i_p668      P=668  init 20.6103 -> step3000 59.2088   drift +187.3%   params 1,368,064
  [PASS] sp_i_p668: final drift is +187.3% - +187.3%
  sp_ii_p668     P=668  init 20.6103 -> step3000 59.8068   drift +190.2%   params 1,368,064
  [PASS] sp_ii_p668: final drift is +190.2% - +190.2%
  sp_iii_p139    P=139  init 32.0331 -> step3000 180.0249   drift +462.0%   params 2,562,048
  [PASS] sp_iii_p139: final drift is +462.0% - +462.0%


## 9. Score-file integrity and key alignment

Every quantitative claim joins on `uid` (domain) or `key` (OmniContext). Two closed
artifacts predate that rule and carry only `i`; two further files are embedding metrics
and carry no PF/SC by design.

In [12]:
files = sorted(glob.glob(f"{SC}/*.jsonl"))
n_err = n_rng = 0
no_key, no_pfsc = [], []
for fn in files:
    rs = list(rows(fn))
    if not rs:
        continue
    b = os.path.basename(fn)
    if "uid" not in rs[0] and "key" not in rs[0]:
        no_key.append(b)
    if "PF" not in rs[0] or "SC" not in rs[0]:
        no_pfsc.append(b)
        continue
    for r in rs:
        if "error" in r:
            n_err += 1
        for k in ("PF", "SC"):
            v = r.get(k)
            if not isinstance(v, (int, float)) or not 0 <= v <= 10:
                n_rng += 1

print(f"  {len(files)} files | rows with an 'error' key {n_err} | bad PF/SC {n_rng}")
print(f"  no uid/key: {no_key}")
print(f"  no PF/SC (embedding metrics): {no_pfsc}")
check("no scored row carries an error", n_err == 0)
check("all PF/SC numeric and within [0,10]", n_rng == 0)
check("only the closed artifacts lack uid/key",
      set(no_key) <= {"baseline_v1.jsonl", "sd_reps.jsonl"}, str(no_key))

def alignment(pattern, field):
    ref, mism, n = None, 0, 0
    for fn in sorted(glob.glob(f"{SC}/{pattern}")):
        rs = list(rows(fn))
        if not rs or field not in rs[0]:
            continue
        m = {r["i"]: r[field] for r in rs}
        n += 1
        if ref is None:
            ref = m
        else:
            mism += sum(1 for i in m if ref.get(i) != m[i])
    return n, mism

n, mism = alignment("test__*.jsonl", "uid")
print(f"  i->uid identical across {n} test score files: {mism} mismatches")
check("i->uid alignment on the test split", mism == 0)

n, mism = alignment("omnicontext__*.jsonl", "key")
print(f"  i->key identical across {n} omnicontext score files: {mism} mismatches")
check("i->key alignment on OmniContext", mism == 0)

  91 files | rows with an 'error' key 0 | bad PF/SC 0
  no uid/key: ['baseline_v1.jsonl', 'sd_reps.jsonl']
  no PF/SC (embedding metrics): ['baseline_arcface.jsonl', 'baseline_embed.jsonl', 'screen__baseline__rerun_similarity.jsonl', 'screen__baseline_similarity.jsonl', 'screen__step100_similarity.jsonl', 'screen__step500_similarity.jsonl', 'test__baseline_identity.jsonl', 'test__baseline_target.jsonl', 'test__lora_r8_step1000_identity.jsonl', 'test__lora_r8_step1000_target.jsonl', 'test__lora_r8_step4000_identity.jsonl', 'test__lora_r8_step4000_target.jsonl', 'test__qlora_r8_step2000_identity.jsonl', 'test__qlora_r8_step2000_target.jsonl', 'test__sp_iii_p139_step2000_identity.jsonl', 'test__sp_iii_p139_step2000_target.jsonl']
  [PASS] no scored row carries an error
  [PASS] all PF/SC numeric and within [0,10]
  [PASS] only the closed artifacts lack uid/key - ['baseline_v1.jsonl', 'sd_reps.jsonl']
  i->uid identical across 30 test score files: 0 mismatches
  [PASS] i->uid alignment on 

### 9b. Collinearity of the stratification variables

Reported alongside every stratified delta; recorded in no notebook output until now.

In [13]:
def cramers_v(a, b):
    ca, cb = sorted(set(a)), sorted(set(b))
    tab = np.zeros((len(ca), len(cb)))
    for x, y in zip(a, b):
        tab[ca.index(x)][cb.index(y)] += 1
    chi2 = st.chi2_contingency(tab, correction=False)[0]
    return math.sqrt(chi2 / (tab.sum() * (min(tab.shape) - 1))), tab, ca, cb

base = load_scores("test__baseline.jsonl")
v, tab, ca, cb = cramers_v([r["ref_kind"] for r in base], [r["n_refs"] for r in base])
print(f"  Cramer's V(ref_kind, n_refs) on test = {v:.3f}")
for i, k in enumerate(ca):
    print(f"    {k:<8} " + "  ".join(f"{cb[j]}-ref {int(tab[i][j]):>3}" for j in range(len(cb))))

  Cramer's V(ref_kind, n_refs) on test = 0.454
    all_up   1-ref 127  2-ref  64  3-ref   7
    mixed    1-ref   0  2-ref  94  3-ref  39
    no_up    1-ref 116  2-ref  36  3-ref   5


## 10. Environment, engine consistency and split bookkeeping

Reads the 68 generation manifests, the generation notebooks themselves, and the split files.
Establishes what every cross-arm comparison rests on: one engine and one set of sampler
settings, with the three intended exceptions named. All recorded repository commit values
agree, but commit provenance is present on only a subset of the manifests (10a).

In [14]:
ROOT = os.path.dirname(DRIVE)
MAN = sorted(glob.glob(f"{DRIVE}/gen/**/manifest.json", recursive=True))
manifests = {m.split("/gen/")[1].rsplit("/manifest.json", 1)[0]: json.load(open(m)) for m in MAN}
for k in sorted(manifests):
    if k.startswith("screen/"):
        print(k)
print(f"generation manifests: {len(manifests)}")
for leg in ("screen", "test", "val", "omnicontext"):
    n = sum(1 for k in manifests if k.startswith(leg + "/"))
    print(f"  {leg:<12} {n}")
check("68 generation manifests present", len(manifests) == 68, str(len(manifests)))

screen/a1_complete
screen/a2_identity
screen/a3_both
screen/a4_style
screen/a5_filler
screen/baseline
screen/baseline__rerun
screen/n_contrastive
screen/n_contrastive2
screen/n_empty
screen/n_filler
screen/positive_control_r8_step100
screen/positive_control_r8_step500
generation manifests: 68
  screen       13
  test         17
  val          33
  omnicontext  5
  [PASS] 68 generation manifests present - 68


### 10a. Commit, dataset revision and environment

`repo_commit` and `dataset_revision` are recorded on the runs that need them rather than on
all 68. Where present they must agree.

In [15]:
def field(d, *path, default=None):
    for p in path:
        if not isinstance(d, dict) or p not in d:
            return default
        d = d[p]
    return d

for key in ("repo_commit", "dataset_revision", "torch", "transformers", "gpu"):
    vals = {}
    for run, d in manifests.items():
        v = d.get(key)
        if v is not None:
            vals.setdefault(v, []).append(run)
    present = sum(len(v) for v in vals.values())
    print(f"  {key:<18} recorded on {present}/68")
    for v, runs in sorted(vals.items(), key=lambda kv: -len(kv[1])):
        print(f"      {str(v)[:56]:<58} {len(runs)} run(s)")

commits = {d["repo_commit"] for d in manifests.values() if d.get("repo_commit")}
revs = {d["dataset_revision"] for d in manifests.values() if d.get("dataset_revision")}
print(f"\n  distinct repo_commit values: {commits or 'none recorded'}")
print(f"  distinct dataset_revision values: {revs or 'none recorded'}")
check("all recorded repo_commit values agree", len(commits) <= 1, str(commits))
check("all recorded dataset_revision values agree", len(revs) <= 1, str(revs))

gpus = {}
for run, d in manifests.items():
    gpus.setdefault(d.get("gpu", "unrecorded"), []).append(run.split("/")[0])
print("\n  GPU by leg:")
for g, legs in gpus.items():
    from collections import Counter as _C
    print(f"      {g:<26} {dict(_C(legs))}")

  repo_commit        recorded on 2/68
      18e6f9d5271b517fcb32e999f10df943ae9b8f20                   2 run(s)
  dataset_revision   recorded on 5/68
      0f6aabbe8d3448d51184093372e980205d9980d3                   5 run(s)
  torch              recorded on 46/68
      2.6.0+cu124                                                46 run(s)
  transformers       recorded on 2/68
      4.51.3                                                     2 run(s)
  gpu                recorded on 68/68
      NVIDIA A100-SXM4-80GB                                      61 run(s)
      NVIDIA A100-SXM4-40GB                                      7 run(s)

  distinct repo_commit values: {'18e6f9d5271b517fcb32e999f10df943ae9b8f20'}
  distinct dataset_revision values: {'0f6aabbe8d3448d51184093372e980205d9980d3'}
  [PASS] all recorded repo_commit values agree - {'18e6f9d5271b517fcb32e999f10df943ae9b8f20'}
  [PASS] all recorded dataset_revision values agree - {'0f6aabbe8d3448d51184093372e980205d9980d3'}

  GPU by l

### 10b. Engine settings across all 68 runs

Every entry must be constant except three, all intended: `omnicontext/baseline_shipped`
generates at the released geometry, `screen/n_empty` is the arm whose negative prompt is
dropped, and the P=668 soft-prompt runs extend the rotary table. 10d re-derives the bound
that extension protects against.

In [16]:
SHIPPED_ROPE = (1024, 1664, 1664)          # transformer/config.json axes_lens

def settings(d):
    s = d.get("sampler", {})
    g = d.get("geometry", {})
    return {"seed": s.get("seed"), "steps": s.get("steps"),
            "text_g": s.get("text_guidance_scale"), "image_g": s.get("image_guidance_scale"),
            "cfg_range": tuple(s.get("cfg_range", [])),
            "max_seq": s.get("max_sequence_length"),
            "size": s.get("size", g.get("width")),
            "align_res": s.get("align_res", g.get("align_res")),
            "rope": tuple(d.get("rope_axes_lens", SHIPPED_ROPE)),
            "neg": d.get("negative_prompt", "")}

S = {run: settings(d) for run, d in manifests.items()}

# the three intended deviations, and the three lever-2 arms that append to the shipped negative
EXPECTED_GEOM = {"omnicontext/baseline_shipped"}
# the P=668 soft-prompt runs extend the rope table: cap_seq_len is text_len + P and the axis-0
# index also advances by max(ref_H_tokens, ref_W_tokens) per reference, reaching 1009 on test
# and 988 on val against a released axes_lens[0] of 1024. Extension is inert below the original
# length: get_1d_rotary_pos_embed frequencies depend on (dim, theta) only.
EXPECTED_ROPE_DEV = {f"{leg}/sp_{v}_p668__step{st}"
                     for leg, steps in (("val", (500, 1000, 2000, 3000)), ("test", (2000,)))
                     for v in ("i", "ii") for st in steps}
EXPECTED_EMPTY_NEG = {"screen/n_empty"}
EXPECTED_APPENDED_NEG = {"screen/n_contrastive", "screen/n_contrastive2", "screen/n_filler"}

for f in ("seed", "steps", "text_g", "image_g", "cfg_range", "max_seq"):
    vals = {s[f] for s in S.values()}
    print(f"  {f:<10} {vals}")
    check(f"{f} constant across all 68 runs", len(vals) == 1, str(vals))

geom_dev = {r for r, s in S.items() if s["size"] != 512 or s["align_res"] is not False}
print(f"\n  size/align_res deviations: {geom_dev or 'none'}")
check("only baseline_shipped deviates on geometry", geom_dev == EXPECTED_GEOM, str(geom_dev))

rope_dev = {r for r, s in S.items() if s["rope"] != SHIPPED_ROPE}
print(f"  rope_axes_lens values: {sorted({s['rope'] for s in S.values()})}")
print(f"  rope deviations: {len(rope_dev)} runs")
check("only the P=668 soft-prompt runs extend the rotary table",
      rope_dev == EXPECTED_ROPE_DEV, str(rope_dev ^ EXPECTED_ROPE_DEV))

# the shipped negative is the modal value; lever 2 appends to it, n_empty drops it
from collections import Counter as _C
SHIPPED = _C(s["neg"] for s in S.values()).most_common(1)[0][0]
empty = {r for r, s in S.items() if not s["neg"]}
appended = {r for r, s in S.items() if s["neg"] and s["neg"] != SHIPPED}
not_prefixed = {r for r in appended if not S[r]["neg"].startswith(SHIPPED)}
print(f"  runs on the shipped negative: {sum(1 for s in S.values() if s['neg'] == SHIPPED)}/68")
print(f"  negative dropped:  {empty or 'none'}")
print(f"  negative appended: {sorted(appended) or 'none'}")
check("only n_empty drops the negative prompt", empty == EXPECTED_EMPTY_NEG, str(empty))
check("only the three lever-2 arms append to it", appended == EXPECTED_APPENDED_NEG, str(appended))
check("every appended negative starts with the shipped one", not not_prefixed, str(not_prefixed))

rows_expected = {"screen": 302, "test": 488, "val": 98, "omnicontext": 400}
bad = {r: (d.get("n_images") or d.get("rows")) for r, d in manifests.items()
       if (d.get("n_images") or d.get("rows")) != rows_expected[r.split("/")[0]]}
print(f"  row counts off their leg: {bad or 'none'}")
check("every run generated its full leg", not bad, str(bad))

  seed       {0}
  [PASS] seed constant across all 68 runs - {0}
  steps      {50}
  [PASS] steps constant across all 68 runs - {50}
  text_g     {5.0}
  [PASS] text_g constant across all 68 runs - {5.0}
  image_g    {2.0}
  [PASS] image_g constant across all 68 runs - {2.0}
  cfg_range  {(0.0, 1.0)}
  [PASS] cfg_range constant across all 68 runs - {(0.0, 1.0)}
  max_seq    {1024}
  [PASS] max_seq constant across all 68 runs - {1024}

  size/align_res deviations: {'omnicontext/baseline_shipped'}
  [PASS] only baseline_shipped deviates on geometry - {'omnicontext/baseline_shipped'}
  rope_axes_lens values: [(1024, 1664, 1664), (4096, 4096, 4096)]
  rope deviations: 10 runs
  [PASS] only the P=668 soft-prompt runs extend the rotary table - set()
  runs on the shipped negative: 64/68
  negative dropped:  {'screen/n_empty'}
  negative appended: ['screen/n_contrastive', 'screen/n_contrastive2', 'screen/n_filler']
  [PASS] only n_empty drops the negative prompt - {'screen/n_empty'}
  [PASS] 

### 10c. One engine, in code

The manifests declare the settings. This checks the code that produced them: the reference
preprocessing and the sampler call are extracted from every notebook that defines them and
compared by exact text.

In [17]:
def fn_source(src, name):
    out, grab = [], False
    for line in src.splitlines():
        if line.startswith(f"def {name}("):
            grab, out = True, [line]
            continue
        if grab:
            if line.strip() and not line.startswith((" ", "\t")):
                break
            out.append(line.rstrip())
    return "\n".join(out).rstrip() if out else None

def strip_comments(s):
    keep = []
    for l in s.splitlines():
        l = re.sub(r"\s+#.*$", "", l) if "#" in l else l
        if l.strip():
            keep.append(l.rstrip())
    return "\n".join(keep)

def sampler_defaults(body):
    try:
        fd = ast.parse(body).body[0]
    except SyntaxError:
        return None
    args = [a.arg for a in fd.args.args]
    defs = fd.args.defaults
    m = dict(zip(args[len(args) - len(defs):], [getattr(x, "value", None) for x in defs]))
    return tuple((k, m.get(k)) for k in ("seed", "steps", "text_g", "image_g", "size"))

import ast, re
nbs = [p for p in sorted(glob.glob(f"{ROOT}/**/*.ipynb", recursive=True))
       if ".ipynb_checkpoints" not in p]
print(f"notebooks scanned: {len(nbs)}")

prep, samp = {}, {}
for p in nbs:
    try:
        d = json.load(open(p, encoding="utf-8"))
    except Exception:
        continue
    src = "\n".join("".join(c["source"]) for c in d.get("cells", []) if c["cell_type"] == "code")
    b = fn_source(src, "load_inputs")
    if b:
        prep.setdefault(hashlib.md5(strip_comments(b).encode()).hexdigest(), []).append(os.path.basename(p))
    for name in ("generate", "generate_with"):
        g = fn_source(src, name)
        if not g:
            continue
        k = sampler_defaults(g)
        if k:
            samp.setdefault(k, []).append(os.path.basename(p))

print(f"\n  def load_inputs, comments stripped: {sum(len(v) for v in prep.values())} definitions, "
      f"{len(prep)} distinct")
for h, files in sorted(prep.items(), key=lambda kv: -len(kv[1])):
    print(f"      {h[:8]}  {len(files):>3}" + ("" if len(files) > 3 else "  " + ", ".join(files)))

print(f"\n  sampler defaults on generate / generate_with:")
for k, files in sorted(samp.items(), key=lambda kv: -len(kv[1])):
    print(f"      {dict(k)}  {len(files):>3}" + ("" if len(files) > 3 else "  " + ", ".join(files)))

# baseline_shipped passes references unresized at the released geometry; omnigen2_test is the
# first pipeline smoke test and produced no scored arm
prep_dev = {f for h, files in prep.items() if len(files) < 3 for f in files}
main = max(samp, key=lambda k: len(samp[k]))
samp_dev = {f for k, files in samp.items() if k != main for f in files}
check("reference preprocessing is one implementation outside baseline_shipped",
      prep_dev == {"omnicontext_gen_baseline_shipped.ipynb"}, str(sorted(prep_dev)))
check("sampler defaults agree outside baseline_shipped and the smoke test",
      samp_dev == {"omnicontext_gen_baseline_shipped.ipynb", "omnigen2_test.ipynb"},
      str(sorted(samp_dev)))
print(f"\n  {len(samp[main])} notebooks share {dict(main)}")

notebooks scanned: 153

  def load_inputs, comments stripped: 45 definitions, 2 distinct
      480358dd   44
      86a10031    1  omnicontext_gen_baseline_shipped.ipynb

  sampler defaults on generate / generate_with:
      {'seed': 0, 'steps': 50, 'text_g': 5.0, 'image_g': 2.0, 'size': 512}   44
      {'seed': 0, 'steps': 50, 'text_g': 5.0, 'image_g': 2.0, 'size': 1024}    1  omnicontext_gen_baseline_shipped.ipynb
      {'seed': 0, 'steps': 20, 'text_g': 4.0, 'image_g': 2.0, 'size': 512}    1  omnigen2_test.ipynb
  [PASS] reference preprocessing is one implementation outside baseline_shipped - ['omnicontext_gen_baseline_shipped.ipynb']
  [PASS] sampler defaults agree outside baseline_shipped and the smoke test - ['omnicontext_gen_baseline_shipped.ipynb', 'omnigen2_test.ipynb']

  44 notebooks share {'seed': 0, 'steps': 50, 'text_g': 5.0, 'image_g': 2.0, 'size': 512}


### 10d. Rotary bound, re-derived

The soft-prompt generation notebooks assert cap_seq_len <= axes_lens[0]. cap_seq_len is
attention_mask.sum(), the conditioning length. The axis-0 index the model assigns is larger:
OmniGen2RotaryPosEmbed.forward sets pe_shift = cap_seq_len, then advances it by
max(ref_H_tokens, ref_W_tokens) for every reference before positioning the output image.
At size 512 with patch_size 2 that is 512 // 8 // 2 = 32 per reference. The bound below pairs
the longest conditioning stream with the largest reference count on the split, so it is an
upper bound on the true maximum.

In [18]:
REF_TOKENS = 512 // 8 // 2

cap_printed = {}
for p in nbs:
    b = os.path.basename(p)
    if not re.match(r"soft_prompt_(test|val)_gen_(i|ii|iii)\.ipynb$", b):
        continue
    d = json.load(open(p, encoding="utf-8"))
    txt = "\n".join("".join(o.get("text", []))
                    for c in d.get("cells", []) for o in c.get("outputs", [])
                    if o.get("output_type") == "stream")
    m = re.search(r"max cap_seq_len = (\d+)", txt)
    if m:
        cap_printed[b] = int(m.group(1))

max_refs = {leg: max(len(r["input_images"]) for r in rows(f"{DRIVE}/{leg}.jsonl"))
            for leg in ("test", "val")}

bounds = {}
for b, cap in sorted(cap_printed.items()):
    leg = "test" if "_test_" in b else "val"
    bounds[b] = cap + REF_TOKENS * max_refs[leg]
    print(f"  {b:<32} cap_seq_len {cap:>4} + {REF_TOKENS}x{max_refs[leg]} -> axis-0 <= {bounds[b]:>4}")

check("no evaluation row reaches the shipped rope table's 1024",
      max(bounds.values()) < SHIPPED_ROPE[0], f"max {max(bounds.values())}")
check("the extended table is never the reason a run succeeded",
      max(bounds.values()) < 4096, f"max {max(bounds.values())}")

  soft_prompt_test_gen_i.ipynb     cap_seq_len  913 + 32x3 -> axis-0 <= 1009
  soft_prompt_test_gen_ii.ipynb    cap_seq_len  913 + 32x3 -> axis-0 <= 1009
  soft_prompt_test_gen_iii.ipynb   cap_seq_len  245 + 32x3 -> axis-0 <=  341
  soft_prompt_val_gen_i.ipynb      cap_seq_len  892 + 32x3 -> axis-0 <=  988
  soft_prompt_val_gen_ii.ipynb     cap_seq_len  892 + 32x3 -> axis-0 <=  988
  soft_prompt_val_gen_iii.ipynb    cap_seq_len  224 + 32x3 -> axis-0 <=  320
  [PASS] no evaluation row reaches the shipped rope table's 1024 - max 1009
  [PASS] the extended table is never the reason a run succeeded - max 1009


### 10e. Checkpoint provenance for the runs whose directory name carries no step

`gen/test/lora_r{4,8,16}` were produced at step 4000, but neither the directory name nor the
manifest records it; the step is pinned only by the notebook that wrote them. Recovered here
so the step-4000 arm rests on an artifact rather than on reading the source.

In [19]:
lora_step = {}
for p in nbs:
    b = os.path.basename(p)
    if not re.match(r"lora_arm(_step\d+)?_r\d+\.ipynb$", b):
        continue
    src = "\n".join("".join(c["source"]) for c in json.load(open(p, encoding="utf-8"))["cells"]
                    if c["cell_type"] == "code")
    ms, mr = re.search(r"(?m)^STEP\s*=\s*(\d+)", src), re.search(r"(?m)^RANK\s*=\s*(\d+)", src)
    if ms and mr:
        lora_step[b] = (int(mr.group(1)), int(ms.group(1)), "checkpoint-{STEP}" in src)

disagree = []
for b, (rank, step, pinned) in sorted(lora_step.items()):
    run = f"lora_r{rank}" if "_step" not in b else f"lora_r{rank}_step{step}"
    man = manifests.get(f"test/{run}", {})
    if not pinned or ("step" in man and man["step"] != step):
        disagree.append(b)
    print(f"  {b:<28} rank {rank:>2} step {step:>4} -> gen/test/{run}"
          f"{'' if 'step' in man else '   (step not in manifest)'}")

check("every lora_arm notebook pins its checkpoint and agrees with its manifest",
      not disagree, str(disagree))

  lora_arm_r16.ipynb           rank 16 step 4000 -> gen/test/lora_r16   (step not in manifest)
  lora_arm_r4.ipynb            rank  4 step 4000 -> gen/test/lora_r4   (step not in manifest)
  lora_arm_r8.ipynb            rank  8 step 4000 -> gen/test/lora_r8   (step not in manifest)
  lora_arm_step1000_r16.ipynb  rank 16 step 1000 -> gen/test/lora_r16_step1000
  lora_arm_step1000_r4.ipynb   rank  4 step 1000 -> gen/test/lora_r4_step1000
  lora_arm_step1000_r8.ipynb   rank  8 step 1000 -> gen/test/lora_r8_step1000
  [PASS] every lora_arm notebook pins its checkpoint and agrees with its manifest - []


### 10f. Recorded selection strings against the actual argmax

In [20]:
sel_bad = []
for run, d in sorted(manifests.items()):
    if "selection" not in d or "step" not in d:
        continue
    leg = "val__" + run.split("/")[1].split("__")[0]
    g = {}
    for f in glob.glob(f"{DRIVE}/scores/{leg}_step*.jsonl"):
        st = int(re.search(r"_step(\d+)\.jsonl$", f).group(1))
        g[st] = float(np.mean([ov(r) for r in rows(f)]))
    if not g:
        continue
    unrestricted = max(g, key=g.get)
    post = {s: v for s, v in g.items() if s > 500}          # warmup_t = 500 in every arm
    restricted = max(post, key=post.get) if post else None
    ok = (d["step"] == restricted) if "post-warmup" in d["selection"] else (d["step"] == unrestricted)
    print(f"  {run:<38} step {d['step']:>5}  unrestricted {unrestricted:>5}  post-warmup {str(restricted):>5}  {'ok' if ok else 'STRING OVERSTATES'}")
    if not ok:
        sel_bad.append(run)

check("only the two documented SP-iii manifests contradict the unrestricted val argmax",
      set(sel_bad) == {"test/sp_iii_p139__step2000", "omnicontext/sp_iii_p139__step2000"},
      f"{sel_bad}")
print("  the two SP-iii manifests say 'pre-registered' over 500/1000/2000/3000 while using the")
print("  post-warmup argmax. Frozen artifacts are not edited; the restriction is documented in")
print("  soft_prompt_test_gen_iii and the notebooks now state the restricted rule.")

  omnicontext/qlora_r8__step2000         step  2000  unrestricted  2000  post-warmup  2000  ok
  omnicontext/sp_iii_p139__step2000      step  2000  unrestricted   500  post-warmup  2000  STRING OVERSTATES
  test/qlora_r8__step2000                step  2000  unrestricted  2000  post-warmup  2000  ok
  test/sp_i_p668__step2000               step  2000  unrestricted  2000  post-warmup  2000  ok
  test/sp_ii_p668__step2000              step  2000  unrestricted  2000  post-warmup  2000  ok
  test/sp_iii_p139__step2000             step  2000  unrestricted   500  post-warmup  2000  STRING OVERSTATES
  [PASS] only the two documented SP-iii manifests contradict the unrestricted val argmax - ['omnicontext/sp_iii_p139__step2000', 'test/sp_iii_p139__step2000']
  the two SP-iii manifests say 'pre-registered' over 500/1000/2000/3000 while using the
  post-warmup argmax. Frozen artifacts are not edited; the restriction is documented in
  soft_prompt_test_gen_iii and the notebooks now state the restri

### 10g. Screen split and the effective training set

The screen slice is carved from train by target and excluded from every training run, so the
rows actually trained on are fewer than `train.jsonl` carries.

In [21]:
screen = list(rows(f"{DRIVE}/screen.jsonl"))
excl = {l.strip() for l in open(f"{DRIVE}/screen_exclude_from_training.txt") if l.strip()}
scr_t = {r["output_image"] for r in screen}

tr_sorted = sorted(frozen["train"])
random.seed(42)
random.shuffle(tr_sorted)
repro = set(tr_sorted[:len(scr_t)])

in_train = [r for r in splits["train"] if r["output_image"] in excl]
effective = len(splits["train"]) - len(in_train)

print(f"  screen: {len(screen)} rows, {len(scr_t)} targets")
print(f"  exclude list entries: {len(excl)}")
print(f"  train rows belonging to the excluded targets: {len(in_train)}")
print(f"  train.jsonl {len(splits['train'])} - screen {len(in_train)} = {effective} rows actually trained on")

check("screen targets reproduce from sorted(train targets), seed 42", repro == scr_t)
check("exclude list is exactly the screen target set", excl == scr_t)
check("screen is a subset of train, disjoint from val and test",
      scr_t <= frozen["train"] and not (scr_t & frozen["test"]) and not (scr_t & frozen["val"]))
check("effective training set is 39,591 rows", effective == 39591, str(effective))

for lbl, rs, n in (("screen", screen, len(screen)), ("test", splits["test"], len(splits["test"]))):
    from collections import Counter as _C
    mix = _C(len(r["input_images"]) for r in rs)
    print(f"  {lbl} reference mix: " + " | ".join(
        f"{k}-ref {mix[k]} ({100*mix[k]/n:.1f}%)" for k in sorted(mix)))

  screen: 302 rows, 62 targets
  exclude list entries: 62
  train rows belonging to the excluded targets: 302
  train.jsonl 39893 - screen 302 = 39591 rows actually trained on
  [PASS] screen targets reproduce from sorted(train targets), seed 42
  [PASS] exclude list is exactly the screen target set
  [PASS] screen is a subset of train, disjoint from val and test
  [PASS] effective training set is 39,591 rows - 39591
  screen reference mix: 1-ref 152 (50.3%) | 2-ref 120 (39.7%) | 3-ref 30 (9.9%)
  test reference mix: 1-ref 243 (49.8%) | 2-ref 194 (39.8%) | 3-ref 51 (10.5%)


### 10h. `ref_kind` against the recorded reference sizes

`ref_kind` is derived in the scorers from the `_up` flag in the filename. The split files
carry the actual pixel dimensions, so the two can be cross-checked without touching an
image.

In [22]:
from collections import Counter as _C
sizes = _C()
mismatch = []
for r in allrows:
    for p, s in zip(r["input_images"], r.get("input_img_sizes", [])):
        t = tuple(s)
        sizes[t] += 1
        up = "_up" in os.path.basename(p)
        if up != (t == (800, 800)):
            mismatch.append((p, t))

print(f"  reference entries: {sum(sizes.values())}")
for t, n in sizes.most_common():
    print(f"      {t[0]}x{t[1]:<6} {n}")
print(f"  filename _up flag vs 800x800 size mismatches: {len(mismatch)}")
check("the _up flag agrees with the recorded size on every reference", not mismatch,
      str(mismatch[:3]))
check("only three reference resolutions exist in the corpus", len(sizes) == 3, str(list(sizes)))

def refkind(r):
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed")

from_jsonl = _C(refkind(r) for r in splits["test"])
from_scores = _C(r["ref_kind"] for r in load_scores("test__baseline.jsonl"))
print(f"  test ref_kind from jsonl   : {dict(from_jsonl)}")
print(f"  test ref_kind from scorers : {dict(from_scores)}")
check("scorer ref_kind matches the split files", from_jsonl == from_scores)

  reference entries: 65043
      800x800    38036
      960x1280   23767
      768x768    3240
  filename _up flag vs 800x800 size mismatches: 0
  [PASS] the _up flag agrees with the recorded size on every reference - []
  [PASS] only three reference resolutions exist in the corpus - [(800, 800), (960, 1280), (768, 768)]
  test ref_kind from jsonl   : {'all_up': 198, 'no_up': 157, 'mixed': 133}
  test ref_kind from scorers : {'all_up': 198, 'no_up': 157, 'mixed': 133}
  [PASS] scorer ref_kind matches the split files


### 10i. Adapter storage dtype

The two converters differ: `lora_convert` rebuilds through peft and writes fp32, while
`qlora_convert` filters and renames the raw checkpoint and keeps the dtype training used.
Both are applied to a bf16 transformer at inference, so the stored dtype does not reach the
forward pass; recorded here because the arms are compared directly.

In [23]:
def dtypes(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        hdr = json.loads(f.read(n))
    return sorted({v["dtype"] for k, v in hdr.items() if k != "__metadata__"})

seen = {}
for p in sorted(glob.glob(f"{EXP}/*/checkpoint-*/transformer_lora/pytorch_lora_weights.safetensors")):
    run = p.split("/experiments/")[1].split("/")[0]
    seen.setdefault(run, set()).update(dtypes(p))
for run, d in sorted(seen.items()):
    print(f"  {run:<16} {sorted(d)}")
check("each run stores its adapter in a single dtype",
      all(len(d) == 1 for d in seen.values()), str({k: sorted(v) for k, v in seen.items()}))

  lora_r16         ['F32']
  lora_r4          ['F32']
  lora_r8          ['F32']
  lora_r8_early    ['F32']
  positive_control_r8 ['F32']
  qlora_r8         ['BF16']
  [PASS] each run stores its adapter in a single dtype - {'lora_r16': ['F32'], 'lora_r4': ['F32'], 'lora_r8': ['F32'], 'lora_r8_early': ['F32'], 'positive_control_r8': ['F32'], 'qlora_r8': ['BF16']}


## 11. Pre-registration integrity

`prereg.txt` is the registration of record for the human leg. The `PREREG` string in
`human_eval_ui` was reworded after the run and no longer matches it. The write is guarded on
the file's absence, so the rewording cannot have reached the file; this section pins that with
the hash, the timestamp ordering and the guard, and names the three divergences.

In [24]:
import datetime, difflib

HE = f"{DRIVE}/human_eval"
PREREG_SHA = "ad588b9fa068eb7e63810ad3aaa52f8f6d619ab8d9c61b4d78cd0645be51ac6c"

blob = open(f"{HE}/prereg.txt", "rb").read()
sha  = hashlib.sha256(blob).hexdigest()
print(f"  prereg.txt {len(blob)} bytes  sha256 {sha[:16]}...")
check("prereg.txt matches the hash recorded in the audit manifest", sha == PREREG_SHA, sha[:16])

txt    = blob.decode("utf-8")
frozen = datetime.datetime.fromisoformat(re.search(r"frozen (\S+)", txt).group(1))
ats    = sorted(r["at"] for r in rows(f"{HE}/responses.jsonl"))
first  = datetime.datetime.fromisoformat(ats[0])
print(f"  frozen {frozen.isoformat()} | first rating {ats[0]} | last {ats[-1]} | lead {first - frozen}")
check("prereg.txt was frozen before the first recorded rating", frozen < first, f"{first - frozen} ahead")

ui  = sorted(glob.glob(f"{ROOT}/**/human_eval_ui.ipynb", recursive=True))[0]
src = "\n".join("".join(c["source"]) for c in json.load(open(ui, encoding="utf-8"))["cells"]
                if c["cell_type"] == "code")
check("human_eval_ui writes prereg.txt only when it does not exist",
      "if not os.path.exists(P):" in src and 'open(P, "w").write(PREREG' in src)

norm = lambda s: [l.strip() for l in s.strip().split("\n") if l.strip() and not l.startswith("frozen ")]
body  = norm(txt)
nbpr  = norm(re.search(r'PREREG = """(.*?)"""', src, re.S).group(1))
diff  = list(difflib.unified_diff(body, nbpr, lineterm="", n=0))
hunks = sum(1 for l in diff if l.startswith("@@"))
lines = [l for l in diff if l[:1] in "+-" and l[:3] not in ("---", "+++")]
print(f"\n  frozen file vs the notebook's PREREG string: {hunks} hunks, {len(lines)} lines")
for l in lines:
    print(f"    {l[0]} {l[1:][:100]}")
MARKERS = ("33/50 needed at p<0.05", "on the non-tie decisions",
           "over all comparisons", "the independent primary comparisons",
           "corroborates", "does not establish equivalence")
found = sum(any(m in l for l in lines) for m in MARKERS)
check("the notebook string diverges from the frozen file in exactly the three documented places",
      hunks == 3 and found == len(MARKERS), f"{hunks} hunks, {found}/{len(MARKERS)} markers")
print("  all three are post-run rewording of the notebook string; the file is untouched:")
print("    per-contrast line drops the fixed 33/50 threshold - execution used the realised")
print("      non-tie denominators 44/23/19 at thresholds 29/17/15")
print("    position-bias line narrows 'all comparisons' to the 150 independent primary pairs -")
print("      consistency is 16-30 (p 0.054) frozen, 11-30 (p 0.0043) restricted, both in human_eval_check")
print("    closing paragraph drops 'corroborates the judge' for a weaker reading")
print("  prereg.txt is echoed verbatim in human_eval_check's own stored output.")

  prereg.txt 1131 bytes  sha256 ad588b9fa068eb7e...
  [PASS] prereg.txt matches the hash recorded in the audit manifest - ad588b9fa068eb7e
  frozen 2026-08-15T14:31:26 | first rating 2026-08-15T15:42:08 | last 2026-08-16T18:14:08 | lead 1:10:42
  [PASS] prereg.txt was frozen before the first recorded rating - 1:10:42 ahead
  [PASS] human_eval_ui writes prereg.txt only when it does not exist

  frozen file vs the notebook's PREREG string: 3 hunks, 8 lines
    - per-contrast win rate over 50 pairs, ties excluded, two-sided binomial, 33/50 needed at p<0.05
    + per-contrast win rate over 50 pairs, ties excluded, two-sided binomial on the non-tie decisions
    - left-preference rate over all comparisons, as a position-bias check, expected near 0.5
    + left-preference rate over the independent primary comparisons, as a position-bias check, expected ne
    - already measured at n=488. A null on LoRA and QLoRA is the predicted result and corroborates
    - the judge; it is not a failed tes

## 12. Val-loss transcriptions

`lora_check`, `qlora_check` and `soft_prompt_check` carry the validation-loss series as
hand-typed dict literals. The literals are read back out of those notebooks and compared
against the `val_loss.jsonl` each was transcribed from.

In [25]:
import ast

SERIES = {   # check notebook -> {VLOSS key: run directory holding val_loss.jsonl}
    "lora_check.ipynb":        {"r4": f"{EXP}/lora_r4", "r8": f"{EXP}/lora_r8", "r16": f"{EXP}/lora_r16"},
    "qlora_check.ipynb":       {None: f"{EXP}/qlora_r8"},
    "soft_prompt_check.ipynb": {"i": f"{SP}/sp_i_p668", "ii": f"{SP}/sp_ii_p668", "iii": f"{SP}/sp_iii_p139"},
}

def vloss_literal(path):
    src = "\n".join("".join(c["source"]) for c in json.load(open(path, encoding="utf-8"))["cells"]
                    if c["cell_type"] == "code")
    i = src.index("VLOSS = {") + len("VLOSS = ")
    depth, j = 0, i
    while True:
        if src[j] == "{":
            depth += 1
        elif src[j] == "}":
            depth -= 1
            if depth == 0:
                break
        j += 1
    return ast.literal_eval(re.sub(r"#[^\n]*", "", src[i:j + 1]))

def source_series(run_dir):
    return {r["step"]: r["val_loss"] for r in rows(f"{run_dir}/val_loss.jsonl")}

bad, total = [], 0
for nb_name, mapping in SERIES.items():
    lit = vloss_literal(sorted(glob.glob(f"{ROOT}/**/{nb_name}", recursive=True))[0])
    for k, run_dir in mapping.items():
        transcribed = lit if k is None else lit[k]
        src = source_series(run_dir)
        miss = [s for s in transcribed if s not in src]
        off  = [(s, round(src[s], 5), transcribed[s]) for s in transcribed
                if s in src and round(src[s], 5) != transcribed[s]]
        total += len(transcribed)
        tag = f"{nb_name.replace('.ipynb', '')}[{k}]" if k else nb_name.replace(".ipynb", "")
        print(f"  {tag:24s} {len(transcribed):2d} pts vs {os.path.basename(run_dir)}/val_loss.jsonl "
              f"({len(src)} logged)  {'exact' if not miss and not off else f'MISSING {miss} OFF {off}'}")
        if miss or off:
            bad.append(tag)
check("every transcribed val-loss point matches its val_loss.jsonl to 5 dp", not bad,
      f"{total} points, {len(bad)} bad")

print()
RUNS = [f"{EXP}/lora_r4", f"{EXP}/lora_r8", f"{EXP}/lora_r16", f"{EXP}/qlora_r8",
        f"{SP}/sp_i_p668", f"{SP}/sp_ii_p668", f"{SP}/sp_iii_p139"]
mono_to_end = []
for run_dir in RUNS:
    s = source_series(run_dir); ks = sorted(s)
    mono = all(s[ks[i + 1]] < s[ks[i]] for i in range(len(ks) - 1))
    print(f"  {os.path.basename(run_dir):14s} argmin step {min(s, key=s.get):>5}  final {s[ks[-1]]:.5f}  "
          f"{'monotone to the last step' if mono else 'flattens, no clear optimum'}")
    if mono:
        mono_to_end.append(os.path.basename(run_dir))
check("val loss falls to the final step in every LoRA rank and in QLoRA, and in no soft-prompt run",
      set(mono_to_end) == {"lora_r4", "lora_r8", "lora_r16", "qlora_r8"}, ", ".join(mono_to_end))

  lora_check[r4]           10 pts vs lora_r4/val_loss.jsonl (16 logged)  exact
  lora_check[r8]           10 pts vs lora_r8/val_loss.jsonl (16 logged)  exact
  lora_check[r16]          10 pts vs lora_r16/val_loss.jsonl (16 logged)  exact
  qlora_check               4 pts vs qlora_r8/val_loss.jsonl (8 logged)  exact
  soft_prompt_check[i]     12 pts vs sp_i_p668/val_loss.jsonl (12 logged)  exact
  soft_prompt_check[ii]    12 pts vs sp_ii_p668/val_loss.jsonl (12 logged)  exact
  soft_prompt_check[iii]   12 pts vs sp_iii_p139/val_loss.jsonl (12 logged)  exact
  [PASS] every transcribed val-loss point matches its val_loss.jsonl to 5 dp - 70 points, 0 bad

  lora_r4        argmin step  4000  final 0.34482  monotone to the last step
  lora_r8        argmin step  4000  final 0.34450  monotone to the last step
  lora_r16       argmin step  4000  final 0.34395  monotone to the last step
  qlora_r8       argmin step  2000  final 0.35466  monotone to the last step
  sp_i_p668      argmin step  27

## 13. Summary

In [26]:
passed = sum(1 for _, p, _ in RESULTS if p)
print(f"{passed}/{len(RESULTS)} checks passed, two of which confine known documented exceptions\n"
      f"rather than asserting clean artifacts: the SP-iii selection-metadata contradiction and the\n"
      f"human_eval_ui PREREG rewording, both below\n")
for name, p, detail in RESULTS:
    print(f"  [{'PASS' if p else 'FAIL'}] {name}" + (f" - {detail}" if detail and not p else ""))
assert passed == len(RESULTS), "one or more artifact checks failed"
print("\nAll artifact checks passed, except the documented SP-iii selection-metadata exception:")
print("  the two SP-iii manifests state the unrestricted argmax rule while recording the")
print("  post-warmup step, which 10f detects and this run treats as a known exception rather")
print("  than a failure.")
print("And the documented PREREG rewording: the human_eval_ui string diverges from the frozen")
print("  prereg.txt in three places, which 11 detects and confines. The file itself is")
print("  hash-verified and predates the first rating.")

64/64 checks passed, two of which confine known documented exceptions
rather than asserting clean artifacts: the SP-iii selection-metadata contradiction and the
human_eval_ui PREREG rewording, both below

  [PASS] subset is 40,479 rows
  [PASS] musical-keyboard count is 408
  [PASS] test split is 7 rows from 1 target
  [PASS] robustness deltas reproduce
  [PASS] target sets reproduce from seed 42
  [PASS] no target overlap between splits
  [PASS] reference spread per split sums to the subset spread
  [PASS] no device keyword in the mismatched pool
  [PASS] mismatched pool disjoint from train/val/test
  [PASS] fewshot_exemplars.jsonl reproduces byte-for-byte
  [PASS] no row exceeds the neutral pool in tokens
  [PASS] sp_i_p668: six prompt tensors, no accelerate state
  [PASS] sp_i_p668: single session, no resume
  [PASS] sp_ii_p668: six prompt tensors, no accelerate state
  [PASS] sp_ii_p668: single session, no resume
  [PASS] sp_iii_p139: six prompt tensors, no accelerate state
  [PASS